In [ ]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

1 Introduction

GeoJSON from test area to RDF Knowledge Graph 

This notebook demonstrates how the testarea from vector data (test_area) can be transformed into an RDF knowledge graph using Python.

It covers the full workflow from loading GeoJSON data with GeoPandas to generating RDF triples with rdflib, including GeoSPARQL geometries and Turtle serialization.


* 2 Load GeoJSON 
* 3 Prepare CRS
* 4 Build RDF Graph
* 5 Add GeoSPARQL Geometries
* 6 Export Turtle
* 7 Example RDF Output


In [ ]:
# Import packages
import geopandas as gpd
from pathlib import Path
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, SKOS

In [ ]:
# Namespaces
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")
GEO  = Namespace("http://www.opengis.net/ont/geosparql#")

In [ ]:
# Import Data

BASE_DIR = Path.cwd().parent  # eine Ebene nach oben

OUTPUT_DIR = BASE_DIR / "data" / "processed"

INPUT_FILE = BASE_DIR / "data" / "processed" / "examples" / "test_area.geojson"

OUTPUT_FILE = BASE_DIR / "data" / "processed" / "test_area_individuals_with_geom.ttl"


In [ ]:
# Set CRS
gdf = gpd.read_file(INPUT_FILE)

if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(4326)

In [ ]:
# Initial Graph

g = Graph()

g.bind("agro", AGRO)
g.bind("geo", GEO)
g.bind("skos", SKOS)
g.bind("rdfs", RDFS)

In [ ]:
# creating RDF

for idx, row in gdf.iterrows():

    area_uri = AGRO[f"test_area_{idx+1}"]
    geom_uri = AGRO[f"geom_test_area_{idx+1}"]

    # Klasse
    g.add((area_uri, RDF.type, AGRO.TestArea))

    # Label
    g.add((
        area_uri,
        SKOS.prefLabel,
        Literal(f"Test Area {idx+1}", lang="en")
    ))

    # ID
    g.add((
        area_uri,
        AGRO.hasID,
        Literal(row["id"])
    ))

    # Geometrie
    g.add((geom_uri, RDF.type, GEO.Geometry))

    g.add((
        geom_uri,
        GEO.asWKT,
        Literal(
            row.geometry.wkt,
            datatype=GEO.wktLiteral
        )
    ))

    g.add((area_uri, GEO.hasGeometry, geom_uri))

In [ ]:
g.serialize(destination=OUTPUT_FILE, format="turtle")

print("Features:", len(gdf))
print("Triples:", len(g))
print("Saved:", OUTPUT_FILE)